# Linear + Softmax による出力生成と Transformer の応用

このノートブックでは、デコーダの **最終段階** である
**Linear + Softmax** による出力トークンの予測を学びます。
また、第4章全体のまとめと Transformer の応用例を紹介します。

書籍 4-16〜4-17 節（図4.61〜4.67）の内容をカバーします。

## 目次
1. デコーダの最終段階の位置づけ（図4.61）
2. Linear: W_out による線形変換（図4.62）
3. Softmax: 確率分布への変換（図4.63）
4. 確率的な予測によるトークン生成（図4.64）
5. Transformer の全体フロー（完全版）
6. 本章の学び（4-17）
7. Transformer の応用例
8. まとめ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 日本語フォント設定（macOS）
plt.rcParams['font.family'] = 'Hiragino Sans'
plt.rcParams['axes.unicode_minus'] = False

# 設定
n_tokens_dec = 3      # デコーダのトークン数
d_model = 6            # 埋め込み次元
n_vocab = 10000        # 語彙数（出力候補のトークン数）

words_dec = ["[BOS]", "富士山", "は"]

print(f"デコーダトークン数: {n_tokens_dec}")
print(f"埋め込み次元 d_model: {d_model}")
print(f"語彙数 (出力候補数): {n_vocab:,}")

## 1. デコーダの最終段階の位置づけ（図4.61）

デコーダの N 層の処理が終わった後、最後に **Linear + Softmax** を通して
次に出力するトークンを決定します。

```
デコーダ出力 (3×6)
    ↓
★ Linear: × W_out (6×10,000)
    ↓
  (3×10,000)
    ↓
★ Softmax: 各行を確率分布に変換
    ↓
  (3×10,000)  ← 各行が 10,000 個の候補トークンの確率
    ↓
  最も確率の高いトークンを選択
```

## 2. Linear: W_out による線形変換（図4.62）

デコーダの出力（3×6）に重みパラメータ行列 $W_{out}$（6×10,000）を掛けます。

$$\underbrace{(3 \times 6)}_{\text{デコーダ出力}} \times \underbrace{(6 \times 10{,}000)}_{W_{out}} = \underbrace{(3 \times 10{,}000)}_{\text{スコア行列}}$$

### この行列の意味

結果の 3×10,000 行列は、3つのトークン（[BOS], 富士山, は）の
それぞれについて、**10,000 種類の候補トークンとの関連性（内積）を計算したもの**です。

値が大きいトークンこそが、次に出力されるトークンだと判断されます。

In [ ]:
# デコーダの出力（ダミーデータ）
np.random.seed(42)
decoder_output = np.round(np.random.randn(n_tokens_dec, d_model) * 0.5, 3)

# W_out (6×10,000)
np.random.seed(123)
W_out = np.random.randn(d_model, n_vocab) * 0.1

print("=== Linear（線形変換）===")
print(f"デコーダ出力: {decoder_output.shape}  (3×6)")
print(f"W_out:        {W_out.shape}  (6×10,000)")

# 行列積
logits = decoder_output @ W_out  # (3×6) × (6×10000) = (3×10000)
print(f"スコア行列:   {logits.shape}  (3×10,000)")
print()
print("各行は 10,000 個の候補トークンとの内積スコア")
print(f"  [BOS]  行のスコア: min={logits[0].min():.3f}, max={logits[0].max():.3f}")
print(f"  富士山 行のスコア: min={logits[1].min():.3f}, max={logits[1].max():.3f}")
print(f"  は     行のスコア: min={logits[2].min():.3f}, max={logits[2].max():.3f}")

## 3. Softmax: 確率分布への変換（図4.63）

スコア行列の各行に Softmax を適用して、**確率分布** に変換します。

$$P(\text{token}_j | \text{context}_i) = \frac{e^{\text{score}_{ij}}}{\sum_{k=1}^{10000} e^{\text{score}_{ik}}}$$

各行の合計が 1 になり、「**確率値に変換する**」ことになります。

In [ ]:
# Softmax で確率分布に変換

def softmax(x):
    """各行に対して Softmax を適用"""
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

probs = softmax(logits)  # (3×10000)

print("=== Softmax（確率分布への変換）===")
print(f"確率行列: {probs.shape}  (3×10,000)")
print()
for i, word in enumerate(words_dec):
    row = probs[i]
    print(f"{word:8s} 行:")
    print(f"  合計 = {row.sum():.6f}  （1になる）")
    print(f"  最大確率 = {row.max():.6f}  (トークン ID: {row.argmax()})")
    print(f"  最小確率 = {row.min():.8f}")
    print()

## 4. 確率的な予測によるトークン生成（図4.64）

各行で **最も確率の高いトークン** を選びます。

| 入力トークン | Softmax 結果 | 選ばれるトークン |
|-------------|-------------|---------------|
| [BOS] | 10,000個の確率値 → 最大のものを選択 | **富士山** |
| 富士山 | 10,000個の確率値 → 最大のものを選択 | **は** |
| は | 10,000個の確率値 → 最大のものを選択 | **春** |

In [ ]:
# 図4.64: 確率的な予測を可視化

# 仮のトークン辞書（上位の確率トークンにラベルを付ける）
# 実際にはランダムな重みなので意味のある結果にはならないが、仕組みを示す
vocab_sample = {0: "[BOS]", 1: "富士山", 2: "は", 3: "春", 4: "に",
                5: "美しい", 6: "。", 7: "[EOS]", 8: "東京", 9: "山"}

# デモ用: 意図的に正しい翻訳結果になるようスコアを調整
np.random.seed(42)
logits_demo = np.random.randn(n_tokens_dec, 10) * 0.3  # 小さめのスコア

# [BOS] → 富士山(1) を最大に
logits_demo[0, 1] = 3.0
# 富士山 → は(2) を最大に  
logits_demo[1, 2] = 3.0
# は → 春(3) を最大に
logits_demo[2, 3] = 3.0

probs_demo = softmax(logits_demo)

print("=== 図4.64: Softmax による確率的予測 ===")
print()
for i, word in enumerate(words_dec):
    print(f'入力「{word}」→ 次のトークンの確率:')
    # 上位5つを表示
    top_indices = np.argsort(probs_demo[i])[::-1][:5]
    for rank, idx in enumerate(top_indices):
        token_name = vocab_sample.get(idx, f'token_{idx}')
        prob = probs_demo[i, idx]
        bar = '█' * int(prob * 50)
        marker = ' ← 選択！' if rank == 0 else ''
        print(f'  {rank+1}位: {token_name:8s} {prob:.4f} {bar}{marker}')
    print()

In [ ]:
# 確率分布の可視化

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (ax, word) in enumerate(zip(axes, words_dec)):
    top_n = 8
    top_indices = np.argsort(probs_demo[i])[::-1][:top_n]
    top_probs = probs_demo[i, top_indices]
    top_labels = [vocab_sample.get(idx, f'ID:{idx}') for idx in top_indices]
    
    colors = ['#e74c3c'] + ['#3498db'] * (top_n - 1)  # 1位だけ赤
    ax.barh(range(top_n), top_probs, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_labels, fontsize=11)
    ax.set_xlabel('確率', fontsize=11)
    ax.set_title(f'入力「{word}」の\n次トークン予測', fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    
    # 確率値を表示
    for j, prob in enumerate(top_probs):
        ax.text(prob + 0.01, j, f'{prob:.3f}', va='center', fontsize=10)

plt.suptitle('図4.64: Softmax による確率的予測（上位8候補）', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("赤色 = 最も確率の高いトークン（次に出力される）")
print("青色 = それ以外の候補")

## 5. Transformer の全体フロー（完全版）

第4章で学んだ Transformer の全処理を振り返ります。

In [ ]:
# Transformer の全体フロー

print("=" * 70)
print("  Transformer の全体フロー（英文和訳の例）")
print("=" * 70)
print()
print('入力: "Mount Fuji looks beautiful in spring."')
print('出力: "富士山は春に美しい。"')
print()
print("── エンコーダ ──────────────────────────────────")
print("  1. Input Embedding:  7単語 → 7×6 行列       (02で学習)")
print("  2. Positional Encoding: 位置情報を加算        (04で学習)")
print("  3. ×N 回繰り返し:")
print("     a. Multi-Head Attention (3ヘッド)          (03で学習)")
print("        Q, K, V 生成 → QKᵀ/√d_k → Softmax → ×V")
print("     b. Add & Norm                             (05で学習)")
print("     c. Feed Forward (6→24→6)                  (06で学習)")
print("     d. Add & Norm")
print("  → エンコーダ出力 (7×6)")
print()
print("── デコーダ ──────────────────────────────────")
print("  1. Output Embedding: [BOS]... → 3×6 行列     (07で学習)")
print("  2. Positional Encoding: 位置情報を加算")
print("  3. ×N 回繰り返し:")
print("     a. Masked Multi-Head Attention              (07で学習)")
print("        Q₁K₁ᵀ/√d_k + M → Softmax → ×V₁")
print("     b. Add & Norm")
print("     c. Cross-Attention                         (07で学習)")
print("        Q=デコーダ, K/V=エンコーダ")
print("     d. Add & Norm")
print("     e. Feed Forward")
print("     f. Add & Norm")
print("  → デコーダ出力 (3×6)")
print()
print("── 最終出力 ──────────────────────────────────")
print("  4. Linear: ×W_out (6×10,000) → (3×10,000)    (★今回学習)")
print("  5. Softmax: 確率分布に変換                     (★今回学習)")
print("  6. 最大確率のトークンを選択 → 次の出力")
print()
print("  これを [EOS] が出力されるまで繰り返す（自己回帰）")

## 6. 本章の学び（4-17）

本章では Transformer を題材として、自然言語処理を実現する AI のメカニズムを
数理的に考察しました。

### Transformer が組み合わせている数理的手法

| 手法 | 使われる場所 |
|------|------------|
| **行列の積** | Q/K/V生成, Attention, Feed Forward, Linear 出力 |
| **内積（ベクトルの類似度）** | QKᵀ（トークン間の関連度）|
| **転置行列** | Kᵀ（行列積を可能にする）|
| **指数関数（$e^x$）** | Softmax |
| **正規化（平均・標準偏差）** | Layer Normalization |
| **sin / cos 関数** | Positional Encoding |
| **ReLU 活性化関数** | Feed Forward |
| **スケーリング（$\sqrt{d_k}$）** | Attention のスコア調整 |

これらを **複合的かつ効果的に組み合わせている** のが Transformer の本質です。

## 7. Transformer の応用例

Transformer は NLP だけでなく、さまざまな分野で活用されています。

### Whisper（音声認識）— 図4.65

OpenAI が開発した音声認識 AI。書き起こしや翻訳などマルチタスクを実行できます。

```
音声波形 → Log-Mel スペクトログラム → 2×Conv1D+GELU
  → Transformer エンコーダ（Self-Attention + MLP）
  → Transformer デコーダ（Cross-Attention でエンコーダ出力を参照）
  → テキスト出力
```

| 構成要素 | 役割 |
|----------|------|
| Log-Mel Spectrogram | 音声を周波数×時間の画像に変換 |
| Conv1D + GELU | 音声の特徴を抽出（GELU は ReLU の仲間）|
| Sinusoidal PE | 時間の位置情報を付与（本章で学んだもの）|
| Learned PE | デコーダ側は学習された位置情報を使用 |

### ViT — Vision Transformer（画像分類）— 図4.66

画像を **パッチ**（小さな正方形）に分割し、Transformer エンコーダで処理します。

```
画像 → 16×16 パッチに分割 → 各パッチをベクトル化
  → Position Embedding を加算
  → Transformer エンコーダ（Self-Attention + MLP）
  → MLP Head → クラス分類
```

| CNN との違い | 説明 |
|-------------|------|
| **CNN** | 畳み込み → 局所的な特徴を学習 |
| **ViT** | Self-Attention → 全パッチ間の関係性を学習（大域的）|

### Stable Diffusion / CLIP（画像生成）— 図4.67

テキストから画像を生成する AI。核心は **拡散モデル（Diffusion Model）** ですが、
テキスト情報の解釈に **CLIP** という Transformer ベースのモデルを使用しています。

```
CLIP の学習:
  テキスト → Text Encoder → テキストベクトル T₁, T₂, ..., Tₙ
  画像    → Image Encoder → 画像ベクトル I₁, I₂, ..., Iₙ
  → 内積で対応関係を学習（対角要素が大きくなるように）
```

Image Encoder には ViT がよく使われています。
テキストと画像のペアデータで学習し、**どのテキスト表現がどの画像表現に対応するか** を学びます。

In [ ]:
# Transformer の応用マップを可視化

fig, ax = plt.subplots(figsize=(14, 7))
ax.axis('off')

# 中央: Transformer
ax.text(0.5, 0.85, 'Transformer', fontsize=20, ha='center', va='center',
        fontweight='bold', bbox=dict(boxstyle='round,pad=0.5', facecolor='#3498db', alpha=0.8),
        color='white')

# NLP
ax.annotate('', xy=(0.15, 0.65), xytext=(0.35, 0.78),
            arrowprops=dict(arrowstyle='->', lw=2, color='#2ecc71'))
ax.text(0.15, 0.58, '自然言語処理\n(NLP)', fontsize=14, ha='center', va='center',
        fontweight='bold', bbox=dict(boxstyle='round', facecolor='#2ecc71', alpha=0.3))
ax.text(0.15, 0.43, '• GPT（文章生成）\n• BERT（文章理解）\n• 機械翻訳', fontsize=10,
        ha='center', va='center')

# 音声
ax.annotate('', xy=(0.5, 0.65), xytext=(0.5, 0.78),
            arrowprops=dict(arrowstyle='->', lw=2, color='#e74c3c'))
ax.text(0.5, 0.58, '音声認識', fontsize=14, ha='center', va='center',
        fontweight='bold', bbox=dict(boxstyle='round', facecolor='#e74c3c', alpha=0.3))
ax.text(0.5, 0.43, '• Whisper\n  （書き起こし・翻訳）\n• エンコーダ+デコーダ', fontsize=10,
        ha='center', va='center')

# 画像
ax.annotate('', xy=(0.85, 0.65), xytext=(0.65, 0.78),
            arrowprops=dict(arrowstyle='->', lw=2, color='#9b59b6'))
ax.text(0.85, 0.58, '画像処理', fontsize=14, ha='center', va='center',
        fontweight='bold', bbox=dict(boxstyle='round', facecolor='#9b59b6', alpha=0.3))
ax.text(0.85, 0.43, '• ViT（画像分類）\n  パッチ → Self-Attention\n• エンコーダのみ', fontsize=10,
        ha='center', va='center')

# マルチモーダル
ax.text(0.5, 0.2, 'マルチモーダル', fontsize=14, ha='center', va='center',
        fontweight='bold', bbox=dict(boxstyle='round', facecolor='#f39c12', alpha=0.3))
ax.text(0.5, 0.07, '• Stable Diffusion / CLIP（テキスト→画像）\n• テキストと画像のペアで学習（内積で対応関係を学ぶ）',
        fontsize=10, ha='center', va='center')
ax.annotate('', xy=(0.35, 0.28), xytext=(0.15, 0.37),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#f39c12', linestyle='--'))
ax.annotate('', xy=(0.65, 0.28), xytext=(0.85, 0.37),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#f39c12', linestyle='--'))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.title('Transformer の応用分野（2024年時点）', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 8. まとめ

### Linear + Softmax による出力

| ポイント | 内容 |
|----------|------|
| **Linear** | $W_{out}$ (6×10,000) を掛けて語彙数分のスコアを計算 |
| **Softmax** | スコアを確率分布に変換（各行の合計=1）|
| **トークン選択** | 最も確率の高いトークンを次の出力として選択 |
| **自己回帰** | 選ばれたトークンを入力に追加し、[EOS] まで繰り返す |

### 第4章 全ノートブック一覧

| No. | テーマ | 主な内容 |
|:---:|--------|----------|
| 01 | Transformer 概要 | エンコーダ・デコーダ構造、Attention の直感 |
| 02 | 単語埋め込み | トークン化、Word Embedding (7×6) |
| 03 | Multi-Head Attention | Q/K/V, 内積スコア, √d_k, Softmax, 3ヘッド結合 |
| 04 | Positional Encoding | sin/cos による位置情報付与 |
| 05 | Add & Norm | Skip Connection, Layer Normalization |
| 06 | Feed Forward | ReLU, 6→24→6 次元変換, N回繰り返し |
| 07 | デコーダ | 自己回帰, Masked MHA, Cross-Attention |
| **08** | **Linear + Softmax** | **最終出力、確率的予測、応用例** |

### 書籍の結びの言葉

> Transformer の基本的構造や数理的な情報処理プロセスを理解しておけば、
> 新出の AI モデルに対しても理解の取っ掛かりとなる部分が少なからずあると
> 実感頂けたと思います。

**第4章の数理的解説は以上です。お疲れ様でした！**